# **Chargement du graphe dans Neo4j Aura**

* Executé depuis VS Code (Colab indisponible : Backend non attribué)

### **IMPORT**

In [1]:
import os
from dotenv import load_dotenv
from pymongo import MongoClient
import certifi
import neo4j

from neo4j import GraphDatabase

In [2]:
load_dotenv()
db = MongoClient(os.getenv("MONGO_URI"))["cinema"]

In [3]:
# certifi fournit une liste à jour des autorités de certification reconnues.
# Avec un certificat personnalisé, le pilote exige le préfixe "neo4j://" (sans "+s")
# et l'option encrypted=True pour garder la connexion chiffrée.
uri = os.getenv("NEO4J_URI").replace("neo4j+s://", "neo4j://")

driver = GraphDatabase.driver(
    uri,
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD")),
    encrypted=True,
    trusted_certificates=neo4j.TrustCustomCAs(certifi.where()),
)
driver.verify_connectivity()
print("Neo4j : connecté")

Neo4j : connecté


In [4]:
contraintes = [
    "CREATE CONSTRAINT film_id IF NOT EXISTS FOR (f:Film) REQUIRE f.id IS UNIQUE",
    "CREATE CONSTRAINT personne_id IF NOT EXISTS FOR (p:Personne) REQUIRE p.id IS UNIQUE",
    "CREATE CONSTRAINT genre_nom IF NOT EXISTS FOR (g:Genre) REQUIRE g.nom IS UNIQUE",
]
with driver.session() as session:
    for c in contraintes:
        session.run(c)
print("Contraintes créées")

Contraintes créées


In [5]:
def charger_par_lots(requete, donnees, taille=1000):
    with driver.session() as session:
        for i in range(0, len(donnees), taille):
            session.run(requete, lignes=donnees[i:i + taille])
    print(len(donnees), "éléments envoyés")

In [6]:
films = [
    {"id": f["_id"], "titre": f["titre"], "annee": f.get("annee"),
     "note": f.get("note_moyenne"), "budget": f.get("budget"),
     "recettes": f.get("recettes"), "genres": f.get("genres", [])}
    for f in db["films"].find()
]

requete_films = """
UNWIND $lignes AS l
MERGE (f:Film {id: l.id})
SET f.titre = l.titre, f.annee = l.annee, f.note = l.note,
    f.budget = l.budget, f.recettes = l.recettes
WITH f, l
UNWIND l.genres AS nom_genre
MERGE (g:Genre {nom: nom_genre})
MERGE (f)-[:A_POUR_GENRE]->(g)
"""
charger_par_lots(requete_films, films)

6015 éléments envoyés


In [7]:
personnes = [
    {"id": p["_id"], "nom": p["nom"], "nb_films": p["nb_films"]}
    for p in db["personnes"].find()
]

requete_personnes = """
UNWIND $lignes AS l
MERGE (p:Personne {id: l.id})
SET p.nom = l.nom, p.nb_films = l.nb_films
"""
charger_par_lots(requete_personnes, personnes)

38865 éléments envoyés


In [8]:
credits = [
    {"film_id": c["film_id"], "personne_id": c["personne_id"],
     "type": c["type"], "role": c.get("role"), "ordre": c.get("ordre")}
    for c in db["credits"].find()
]

requete_credits = """
UNWIND $lignes AS l
MATCH (p:Personne {id: l.personne_id})
MATCH (f:Film {id: l.film_id})
MERGE (p)-[r:A_TRAVAILLE_SUR {type: l.type, role: coalesce(l.role, "")}]->(f)
SET r.ordre = l.ordre
"""
charger_par_lots(requete_credits, credits, taille=5000)

102121 éléments envoyés


In [9]:
with driver.session() as session:
    for label in ["Film", "Personne", "Genre"]:
        n = session.run(f"MATCH (n:{label}) RETURN count(n) AS n").single()["n"]
        print(f"Nœuds {label} : {n}")
    for rel in ["A_TRAVAILLE_SUR", "A_POUR_GENRE"]:
        n = session.run(f"MATCH ()-[r:{rel}]->() RETURN count(r) AS n").single()["n"]
        print(f"Relations {rel} : {n}")

Nœuds Film : 6015
Nœuds Personne : 38865
Nœuds Genre : 19
Relations A_TRAVAILLE_SUR : 102121
Relations A_POUR_GENRE : 15893
